# Radiosonde/Dropsonde Zarr Pipeline
Demonstration of Zarr pipeline using M2HATS data, written for users to understand the process. This will be put into .py files and done behind the scenes. 

---

# 0: Import required packages

In [23]:
from pathlib import Path
import glob
import xarray as xr
import zarr

import glob
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import zarr
from scipy.io import netcdf_file
from scipy.interpolate import interp1d
from tqdm.auto import tqdm

---

# 1: Populate a Zarr store: full-res data

### Create a Zarr store
I created a Zarr store in my scratch directory to contain the individual radiosonde datasets. 

In [2]:
zarr_path = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"
root = zarr.open_group(zarr_path, mode = "a")

### Load data
This M2HATS radiosonde dataset was downloaded from the EOL Field Data Archive as 122 netcdf files, then stored in my scratch directory. I only downloaded ascending files for this case study. 

In [6]:
radiosonde_path = Path("/lustre/desc1/scratch/myasears/sounding_data/m2hats")
radiosonde_files = sorted([p for p in radiosonde_path.iterdir() if p.suffix == ".nc"])

### Populate the Zarr store
I iterate through every netcdf file, open it with xarray, drop incompatible variables, and convert them to Zarr.

In [7]:
for sonde in radiosonde_files:
    sounding_id = sonde.stem
    ds = xr.open_dataset(sonde)
    ds = ds.drop_vars("trajectory") # Drop "trajectory" - data type does not work with Zarr V3
    ds.to_zarr(zarr_path, group=sounding_id, consolidated=False)

### Open Zarr
See whether the Zarr store has been populated -- it has.

In [3]:
m2hats_zarr = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"

In [ ]:
root = zarr.open_group(m2hats_zarr, mode="r")
sounding_ids = list(root.group_keys())

# 2: Populate a Zarr store: gridded data

### Create a Zarr store
I created a Zarr store in my scratch directory to contain the individual radiosonde datasets. 

In [17]:
zarr_path = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats_gridded"
root = zarr.open_group(zarr_path, mode = "a")

### Load data
This M2HATS radiosonde dataset was downloaded from the EOL Field Data Archive as 122 netcdf files, then stored in my scratch directory. I only downloaded ascending files for this case study. 

In [18]:
radiosonde_path = Path("/lustre/desc1/scratch/myasears/sounding_data/m2hats")
radiosonde_files = sorted([p for p in radiosonde_path.iterdir() if p.suffix == ".nc"])

### Set up the grid
To make the most effective Zarr file, the files need to be aligned on a grid. This code establishes a pressure grid.

In [48]:
ds = xr.open_dataset(radiosonde_files[0])
ds = ds.sortby("pres")
ds

<xarray.Dataset> Size: 399kB
Dimensions:         (time: 4748, obs: 1)
Coordinates:
  * time            (time) datetime64[ns] 38kB 2023-07-19T17:32:51 ... 2023-0...
    lat             (time) float32 19kB ...
    lon             (time) float32 19kB ...
    gpsalt          (time) float32 19kB ...
Dimensions without coordinates: obs
Data variables: (12/27)
    trajectory      |S1 1B ...
    launch_time     datetime64[ns] 8B ...
    pres            (time) float32 19kB ...
    tdry            (time) float32 19kB ...
    dp              (time) float32 19kB ...
    rh              (time) float32 19kB ...
    ...              ...
    reference_rh    (obs) float32 4B ...
    reference_wspd  (obs) float32 4B ...
    reference_wdir  (obs) float32 4B ...
    reference_lat   (obs) float32 4B ...
    reference_lon   (obs) float32 4B ...
    reference_alt   (obs) float32 4B ...
Attributes: (12/50)
    Conventions:                           CF-1.6
    RepoRevision:                          V3.4.7
    RepoLastChangedDate:                   Fri May 6 14:29:37 2022 -0600
    RepoId:                                14c38ba5b54ddaa3e2c5c0669fd4f5b756...
    RepoBranch:                            HEAD
    featureType:                           trajectory
    ...                                    ...
    TerminatingAltitude:                   22123 m
    TrackedSatelliteAverageCount:          11.1
    TrefTemperature:                       /////
    TuTemperature:                         20.28 �C
    UCorrection(Uref1-U1):                 -0.13 %Rh
    Uref1Humidity:                         0.00 %Rh

In [36]:
target_pressure = np.arange(1000, 24, -25)  # hPa

In [44]:
def preprocess(ds):

    # Sort by pressure to prep for interpolation
    ds = ds.sortby("pres")
    
    # Interpolate to selected pressure levels
    ds = ds.interp({"pres": target_pressure})
    
    # Drop "trajectory" - data type does not work with Zarr V3
    ds = ds.drop_vars("trajectory")
    
    return ds

In [45]:
ds = xr.open_mfdataset(
    radiosonde_files,
    combine="nested",
    concat_dim="time",
    preprocess=preprocess,
    parallel=True
)

ValueError: Dimensions {'pres'} do not exist. Expected one or more of FrozenMappingWarningOnValuesAccess({'time': 5178, 'obs': 1})

### Populate the Zarr store
I iterate through every netcdf file, open it with xarray, drop incompatible variables, and convert them to Zarr.

In [ ]:
for sonde in radiosonde_files:
    sounding_id = sonde.stem
    ds = xr.open_dataset(sonde)
    ds = ds.drop_vars("trajectory") # Drop "trajectory" - data type does not work with Zarr V3
    ds.to_zarr(zarr_path, group=sounding_id, consolidated=False)